# Alpha diversity

**R equivalent:** `estimate_richness()`, `plot_richness()`

Reproduces the alpha diversity analysis from the phyloseq tutorial on
the GlobalPatterns dataset.

In [ ]:
import numpy as np
import phyla
from phyla import (
    Phyloseq, OtuTable, SampleData, TaxTable,
    estimate_richness, plot_richness,
)
from phyla.testing.fixtures import load_global_patterns_reference, load_golden

ref = load_global_patterns_reference()
gp = Phyloseq(
    otu=OtuTable(ref["otu_table"], taxa_are_rows=True),
    sam=SampleData(ref["sample_data"]),
    tax=TaxTable(ref["tax_table"]),
)
print(gp)

## Compute all richness measures

**R equivalent:** `estimate_richness(gp)`

In [ ]:
richness = estimate_richness(gp)
print(richness.round(2))

## Validate against R reference

In [ ]:
r_ref = load_golden("GlobalPatterns", "estimate_richness")

# Observed richness should match exactly
for sample in richness.index:
    if sample in r_ref.index:
        py_obs = richness.loc[sample, "Observed"]
        r_obs  = r_ref.loc[sample, "Observed"]
        assert abs(py_obs - r_obs) <= 1, (
            f"Observed mismatch for {sample}: phyla={py_obs}, R={r_obs}"
        )

# Shannon within 1%
shared = richness.index.intersection(r_ref.index)
py_shannon = richness.loc[shared, "Shannon"]
r_shannon  = r_ref.loc[shared, "Shannon"].astype(float)
rel_err = ((py_shannon - r_shannon) / r_shannon.replace(0, np.nan)).abs()
assert rel_err.max() < 0.01, f"Shannon max relative error: {rel_err.max():.4f}"

print(f"✓ Observed matches R (all {len(shared)} samples)")
print(f"✓ Shannon max relative error: {rel_err.max():.4f} (threshold: 0.01)")

## Subset measures

**R equivalent:** `estimate_richness(gp, measures=c("Shannon", "Simpson"))`

In [ ]:
r2 = estimate_richness(gp, measures=["Shannon", "Simpson"])
assert list(r2.columns) == ["Shannon", "Simpson"]
print(r2.head())

## Plot richness

**R equivalent:** `plot_richness(gp, x="SampleType", measures=c("Shannon", "Chao1"))`

In [ ]:
p = plot_richness(gp, x="SampleType", measures=["Shannon", "Chao1"])
print(p)